# Manifold Learning

Manifold learning finds a low-dimensional basis for describing high-dimensional data. Manifold learning is a popular approach to nonlinear dimensionality reduction. Algorithms for this task are based on the idea that the dimensionality of many data sets is only artificially high; though each data point consists of perhaps thousands of features, it may be described as a function of only a few underlying parameters. That is, the data points are actually samples from a low-dimensional manifold that is embedded in a high-dimensional space. Manifold learning algorithms attempt to uncover these parameters in order to find a low-dimensional representation of the data.

Some prominent approaches are locally linear embedding (LLE), Hessian LLE, Laplacian eigenmaps, and LTSA. These techniques construct a low-dimensional data representation using a cost function that retains local properties of the data, and can be viewed as defining a graph-based kernel for Kernel PCA. More recently, techniques have been proposed that, instead of defining a fixed kernel, try to learn the kernel using semidefinite programming. The most prominent example of such a technique is maximum variance unfolding (MVU). The central idea of MVU is to exactly preserve all pairwise distances between nearest neighbors (in the inner product space), while maximizing the distances between points that are not nearest neighbors.

An alternative approach to neighborhood preservation is through the minimization of a cost function that measures differences between distances in the input and output spaces. Important examples of such techniques include classical multidimensional scaling (which is identical to PCA), Isomap (which uses geodesic distances in the data space), diffusion maps (which uses diffusion distances in the data space), t-SNE (which minimizes the divergence between distributions over pairs of points), and curvilinear component analysis.

In [26]:
import java.awt.Color;
import java.util.*;
import java.util.stream.*;
import org.apache.commons.csv.CSVFormat;
import smile.feature.extraction.PCA;
import smile.graph.*;
import smile.io.*;
import smile.manifold.*;
import smile.math.MathEx;
import smile.plot.swing.*;
import static smile.swing.SmileUtilities.*;

## Isomap

Isometric feature mapping (isomap) is a widely used low-dimensional embedding methods, where geodesic distances on a weighted graph are incorporated with the classical multidimensional scaling. Isomap is used for computing a quasi-isometric, low-dimensional embedding of a set of high-dimensional data points. Isomap is highly efficient and generally applicable to a broad range of data sources and dimensionalities.

To be specific, the classical MDS performs low-dimensional embedding based on the pairwise distance between data points, which is generally measured using straight-line Euclidean distance. Isomap is distinguished by its use of the geodesic distance induced by a neighborhood graph embedded in the classical scaling. This is done to incorporate manifold structure in the resulting embedding. Isomap defines the geodesic distance to be the sum of edge weights along the shortest path between two nodes. The top n eigenvectors of the geodesic distance matrix, represent the coordinates in the new n-dimensional Euclidean space.
 
The connectivity of each data point in the neighborhood graph is defined as its nearest k Euclidean neighbors in the high-dimensional space. This step is vulnerable to "short-circuit errors" if k is too large with respect to the manifold structure or if noise in the data moves the points slightly off the manifold. Even a single short-circuit error can alter many entries in the geodesic distance matrix, which in turn can lead to a drastically different (and incorrect) low-dimensional embedding. Conversely, if k is too small, the neighborhood graph may become too sparse to approximate geodesic paths accurately.

When the optional parameter CIsomap is true, the method performs C-Isomap that involves magnifying the regions of high density and shrink the regions of low density of data points in the manifold. Edge weights that are maximized in Multi-Dimensional Scaling(MDS) are modified, with everything else remaining unaffected.

In the below example, we apply Isomap to the famous swiss roll data as shown above. This data set was created to test out various dimensionality reduction algorithms. The idea was to create several points in 2d, and then map them to 3d with some smooth function, and then to see what the algorithm would find when it mapped the points back to 2d. The data contains 20,000 samples. Because it is computational intensive to calculate the shortest path for all samples, the example uses only the first 2,000 samples.

In [28]:
var swissroll = Read.csv("base/src/test/resources/data/manifold/swissroll.txt", CSVFormat.DEFAULT.withDelimiter('\t')).toArray();
var figure = ScatterPlot.of(swissroll, '.', Color.BLUE).figure();
show(figure);

we use `k = 7` for neighborhood graph. In the mapped 2d space, we also plot the connections between neighbor neighbors.

In [30]:
var data = Arrays.copyOf(swissroll, 2000);
var nng = NearestNeighborGraph.descent(data, MathEx::distance, 7).largest(false);
var map = IsoMap.fit(nng, new IsoMap.Options(7, 2, false));

var graph = nng.graph(false);
var edges = IntStream.range(0, data.length)
            .mapToObj(graph::getEdges)
            .flatMap(List::stream)
            .map(edge -> new int[]{edge.u(), edge.v()})
            .toArray(int[][]::new);
var figure = Wireframe.of(map, edges).figure();
show(figure);

Note that Isomap produces strange holes like in a slice of Swiss cheese :). This issue can be solved by adding to Isomap a vector quantization step.

## LLE

Locally Linear Embedding (LLE) has several advantages over Isomap, including faster optimization when implemented to take advantage of sparse matrix algorithms, and better results with many problems. LLE also begins by finding a set of the nearest neighbors of each point. It then computes a set of weights for each point that best describe the point as a linear combination of its neighbors. Finally, it uses an eigenvector-based optimization technique to find the low-dimensional embedding of points, such that each point is still described with the same linear combination of its neighbors. LLE tends to handle non-uniform sample densities poorly because there is no fixed unit to prevent the weights from drifting as various regions differ in sample densities.

In [33]:
double[][] coordinates = LLE.fit(data, new LLE.Options(7));

var figure = Wireframe.of(map, edges).figure();
show(figure);

## Laplacian Eigenmap

Using the notion of the Laplacian of the nearest neighbor adjacency graph, Laplacian Eigenmap computes a low dimensional representation of the dataset that optimally preserves local neighborhood information in a certain sense. The representation map generated by the algorithm may be viewed as a discrete approximation to a continuous map that naturally arises from the geometry of the manifold. The parameter t is the smooth/width parameter of heat kernel `exp(-||x-y||2/t)`. Non-positive t means discrete weights.

The locality preserving character of the Laplacian Eigenmap algorithm makes it relatively insensitive to outliers and noise. It is also not prone to "short circuiting" as only the local distances are used.

In [35]:
double[][] coordinates = LaplacianEigenmap.fit(data, new LaplacianEigenmap.Options(7));

var figure = Wireframe.of(map, edges).figure();
show(figure);

## t-SNE

The t-distributed stochastic neighbor embedding (t-SNE) is a nonlinear dimensionality reduction technique that is particularly well suited for embedding high-dimensional data into a space of two or three dimensions, which can then be visualized in a scatter plot. Specifically, it models each high-dimensional object by a two- or three-dimensional point in such a way that similar objects are modeled by nearby points and dissimilar objects are modeled by distant points.

The input data perplexity is the perplexity of the conditional distribution, eta the learning rate, and iterations is the number of iterations. If perplexity is a square matrix, it is assumed to be the distance/dissimilarity matrix.

The t-SNE on the MNIST data is as follows. Note that the input data is preprocessed using PCA to reduce the dimensionality to 50 before t-SNE is performed.

In [37]:
var format = CSVFormat.DEFAULT.withDelimiter(' ');
var mnist = Read.csv("base/src/test/resources/data/mnist/mnist2500_X.txt", format).toArray();
var label = Read.csv("base/src/test/resources/data/mnist/mnist2500_labels.txt", format).column(0).toIntArray();

var pca = PCA.fit(mnist).getProjection(50);
var X = pca.apply(mnist);
var tsne = TSNE.fit(X, new TSNE.Options(2, 20, 200, 12, 550));

var figure = ScatterPlot.of(tsne.coordinates(), label, '@').figure();
figure.setTitle("t-SNE of MNIST");
show(figure);